# Monte Carlo Simulator — Tutorial

This notebook shows basic usage of the monte_carlo.py utilities added to the repository.
It demonstrates: estimating pi, computing expectations, simulating Geometric Brownian Motion (GBM), plotting and saving simulation results, and pricing a European call by Monte Carlo.


## Setup
Run the following cell to install dependencies (only needed if you haven't installed them locally). In most cases you can skip this if you've already run `pip install -r requirements.txt`.


In [ ]:
# Uncomment and run if you need to install requirements in this environment
# !pip install -r requirements.txt


## Imports and helper setup
Import functions from monte_carlo.py and set matplotlib for inline display.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from monte_carlo import estimate_pi, estimate_expectation, simulate_gbm, plot_gbm_paths, price_european_call_mc
print('imports OK')


## Estimate π by Monte Carlo
We sample uniform points in the unit square and compute the fraction inside the quarter circle.


In [ ]:
est, se = estimate_pi(200000)
print(f'pi estimate = {est:.6f}, SE = {se:.6f}')


## Estimate an expectation
Here we estimate E[X^2] for X ~ N(0,1), which theoretically equals 1.


In [ ]:
est, se = estimate_expectation(lambda rng, n: rng.standard_normal(n), lambda x: x * x, 100000)
print(f'est = {est:.6f}, SE = {se:.6f}')


## Simulate GBM and save results
We simulate a set of GBM paths, plot them (inline), save a PNG, and save the simulated data to a compressed NumPy .npz file.


In [ ]:
S0 = 100.0
mu = 0.05
sigma = 0.2
T = 1.0
steps = 252
n_paths = 1000  # reduce for interactive notebook responsiveness
times, paths = simulate_gbm(S0, mu, sigma, T, steps, n_paths)
print('simulated', paths.shape)
# Save data to notebooks/output for this tutorial
import os
os.makedirs('notebooks/output', exist_ok=True)
np.savez_compressed('notebooks/output/gbm_paths.npz', times=times, paths=paths)
print('Saved simulated data to notebooks/output/gbm_paths.npz')
# Plot and save PNG (1980x1770 px mapping at dpi=100)
plot_gbm_paths(times, paths, n_plot=200, figsize_pixels=(1980,1770), dpi=100, save_plot='notebooks/output/gbm.png', show_plot=True)
print('Saved plot to notebooks/output/gbm.png')


## Load saved simulation and inspect
We'll load the .npz file and plot the mean final distribution.


In [ ]:
data = np.load('notebooks/output/gbm_paths.npz')
times = data['times']
paths = data['paths']
print('loaded shapes:', times.shape, paths.shape)
# Plot histogram of terminal values
plt.figure(figsize=(8,4))
plt.hist(paths[:,-1], bins=50, alpha=0.7)
plt.title('Distribution of S_T across simulated paths')
plt.xlabel('S_T')
plt.ylabel('count')
plt.show()


## Price a European call by Monte Carlo
We compute a Monte Carlo price and print the standard error.


In [ ]:
price, se = price_european_call_mc(100, 100, 0.05, 0.2, 1.0, 100000, antithetic=True)
print(f'Price = {price:.6f}, SE = {se:.6f}')


## Next steps
Try increasing the number of paths to see Monte Carlo convergence, add variance reduction methods, or convert the examples into parameter sweeps to create convergence plots.
